# Gaussian splatting on Colab (gsplat, NVIDIA GPU)

Trains `gs_pipeline.py train` on a COLMAP scene made locally (`frames` + `sfm`).

**Before running:** upload to Google Drive (any folder):
- the scene folder, e.g. `local_data/scenes/img_1453/` (needs `images/` and `sparse/0/*.txt`; `database.db` can be left out)
- `src/gaussian_splat/gs_pipeline.py`

Runtime must be a **GPU (T4)** Colab runtime.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU ok:", torch.cuda.is_available())

In [ ]:
# 2. Collect the scene from My Drive (any location) and copy it to the Colab disk as <SCENE>/images + <SCENE>/sparse/0
#    Uses the archive colab_img_1453*.tar.gz if present (it contains everything). Otherwise it assembles the scene from:
#      - the frames folder named IMAGES_DIR
#      - cameras.txt, images.txt, points3D.txt (from local_data/scenes/img_1453/sparse/0/)
#      - gs_pipeline.py
SCENE, IMAGES_DIR = "img_1453", "ps5_images"
import glob, os, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
D, W = '/content/drive/MyDrive', '/content/work'
dst = f'{W}/{SCENE}'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(f'{dst}/sparse/0', exist_ok=True)
find = lambda pat: sorted(glob.glob(f'{D}/**/{pat}', recursive=True))

archives = find(f'colab_{SCENE}*.tar.gz')
if archives:
    print("using archive", archives[0])
    subprocess.run(['tar', 'xzf', archives[0], '-C', W], check=True)  # fails loudly if the upload is incomplete
else:
    imgs = [p for p in find(IMAGES_DIR) if os.path.isdir(p)]
    model = {f: find(f) for f in ('cameras.txt', 'images.txt', 'points3D.txt')}
    script = find('gs_pipeline.py')
    print("frames folder:", imgs, "\nmodel files:", model, "\ngs_pipeline.py:", script)
    missing = ([IMAGES_DIR + '/'] if not imgs else []) + [f for f, v in model.items() if not v] + ([] if script else ['gs_pipeline.py'])
    if missing:
        print("Top-level of My Drive:", os.listdir(D))
        raise SystemExit(f"Missing in Drive: {missing}. Upload them (any folder) and rerun.")
    shutil.copytree(imgs[0], f'{dst}/images')
    for f, v in model.items():
        shutil.copy(v[0], f'{dst}/sparse/0/{f}')
    shutil.copy(script[0], f'{W}/gs_pipeline.py')

# sanity check: every pose in images.txt must have its frame
names = [l.split()[-1] for i, l in enumerate(x for x in open(f'{dst}/sparse/0/images.txt') if not x.startswith('#')) if i % 2 == 0 and l.strip()]
have = set(os.listdir(f'{dst}/images'))
print(f"{dst}: {len(have)} frames, {len(names)} posed images, missing frames: {[n for n in names if n not in have][:5]}")

In [ ]:
# 3. Install gsplat. Its CUDA kernels compile on first use (5-10 min the first time, then cached for the session).
!pip install -q gsplat ninja

In [ ]:
# 4. Train: 30k steps at full resolution (1080x1920) for best quality.
#    If it runs out of GPU memory ("CUDA out of memory"), use --downscale 2 instead.
%cd /content/work
!python gs_pipeline.py train --data {SCENE} --steps 30000 --downscale 1 --out {SCENE}/output

In [ ]:
# 5. Results
print(open(f'/content/work/{SCENE}/output/metrics.txt').read())
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'/content/work/{SCENE}/output/renders/*'))[::8]:
    print(f.split('/')[-1], '(render | ground truth)'); display(Image(f, width=700))

In [ ]:
# 6. Save results to Drive as one archive (download from MyDrive/DVA502/ and unpack into outputs/ locally)
!mkdir -p /content/drive/MyDrive/DVA502
!cd /content/work/{SCENE} && tar czf /content/drive/MyDrive/DVA502/{SCENE}_gsplat_output.tar.gz output
!ls -lh /content/drive/MyDrive/DVA502/